# 🌙 Suivi temporel des alertes — Fink/LSST

Ce notebook trace l'**évolution nuit par nuit** du flux d'alertes Fink/LSST :
- Nombre d'alertes et d'objets nouveaux par nuit, pour chaque tag
- Décomposition par filtre LSST
- Courbe cumulative du nombre d'objets découverts
- Carte de chaleur (heatmap) nuit × filtre
- Détection des nuits les plus actives

**API :** `https://api.lsst.fink-portal.org`  
Endpoints : `/api/v1/tags` · `/api/v1/sources`

## 1. Paramètres

In [ ]:
# ─── Paramètres utilisateur ───────────────────────────────────────────────────
N_PER_TAG  = 2000         # Nombre max d'alertes par tag (grand pour couvrir l'historique)
STARTDATE  = None         # "2025-10-01 00:00:00" ou None
STOPDATE   = None         # "2026-03-15 00:00:00" ou None
BASE_URL   = "https://api.lsst.fink-portal.org"

# Tags à suivre — None = tous les tags disponibles
TAGS_FILTER = None        # ex. ['sn_near_galaxy_candidate', 'in_tns'] ou None

SAVE_FIGS  = True
OUTPUT_DIR = "nightcurve_outputs"
# ─────────────────────────────────────────────────────────────────────────────

## 2. Imports

In [ ]:
import os
import warnings
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.dates as mdates
from IPython.display import display as ipy_display, HTML
from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')
os.makedirs(OUTPUT_DIR, exist_ok=True)

mpl.rcParams.update({
    'font.size'         : 11,
    'axes.titlesize'    : 16,
    'axes.titleweight'  : 'bold',
    'axes.labelsize'    : 16,
    'xtick.labelsize'   : 16,
    'ytick.labelsize'   : 16,
    'legend.fontsize'   : 16,
    'figure.titlesize'  : 13,
    'figure.titleweight': 'bold',
})

FILTER_COLORS = {
    'u': '#7B2FBE',
    'g': '#0077BB',
    'r': '#33AA77',
    'i': '#DDAA33',
    'z': '#BB5500',
    'y': '#AA0000',
}



TAG_PALETTE = [
    '#4477AA', '#EE6677', '#228833', '#CCBB44',
    '#66CCEE', '#AA3377', '#BBBBBB',
]
MJD_EPOCH = pd.Timestamp('1858-11-17', tz='UTC')

def mjd_to_date(mjd):
    """Convertit un scalaire MJD en date (sans heure)."""
    return (MJD_EPOCH + pd.to_timedelta(float(mjd), unit='D')).date()

print("Imports OK ✓")

## 3. Récupération des tags et des alertes

In [ ]:
# ── Liste des tags ────────────────────────────────────────────────────────────
r = requests.get(f"{BASE_URL}/api/v1/tags")
r.raise_for_status()
TAGS_DOC = r.json()
ALL_TAGS = list(TAGS_DOC.keys()) if TAGS_FILTER is None else TAGS_FILTER
print(f"Tags à analyser : {ALL_TAGS}\n")

# ── Téléchargement ────────────────────────────────────────────────────────────
def fetch_tag_data(tag, n, startdate=None, stopdate=None):
    payload = {
        "tag": tag, "n": n, "output-format": "json",
    }
    if startdate: payload["startdate"] = startdate
    if stopdate:  payload["stopdate"]  = stopdate
    resp = requests.post(f"{BASE_URL}/api/v1/tags", json=payload, timeout=60)
    resp.raise_for_status()
    data = resp.json()
    if not data:
        return pd.DataFrame()
    df = pd.DataFrame(data)
    df.columns = [c.replace('r:', '') for c in df.columns]
    for col in ['psfFlux', 'ra', 'dec', 'midpointMjdTai']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    return df


tag_data = {}
for tag in tqdm(ALL_TAGS, desc='Téléchargement', unit='tag'):
    try:
        df = fetch_tag_data(tag, N_PER_TAG, STARTDATE, STOPDATE)
        if not df.empty and 'midpointMjdTai' in df.columns:
            df['night'] = pd.to_datetime(
                df['midpointMjdTai'].dropna().apply(mjd_to_date)
            )
            # Pour les lignes avec MJD manquant
            df['night'] = pd.to_datetime(
                df['midpointMjdTai'].apply(
                    lambda x: mjd_to_date(x) if pd.notna(x) else pd.NaT
                )
            )
        tag_data[tag] = df
        print(f"  ✓  {tag:45s} {len(df):5d} alertes")
    except Exception as e:
        print(f"  ✗  {tag} — {e}")
        tag_data[tag] = pd.DataFrame()

print(f"\n✓ {len(tag_data)} tags chargés.")

## 4. Courbes nuit par nuit — alertes et objets nouveaux

In [ ]:
tags_ok = [t for t, df in tag_data.items() if not df.empty and 'night' in df.columns]
n_tags  = len(tags_ok)

fig, axes = plt.subplots(n_tags, 2, figsize=(16, 3.5 * n_tags),
                          layout='constrained')
if n_tags == 1:
    axes = axes[np.newaxis, :]

fig.suptitle("Évolution nuit par nuit — Fink/LSST", y=1.01)

for row_idx, tag in enumerate(tags_ok):
    df    = tag_data[tag].dropna(subset=['night'])
    color = TAG_PALETTE[row_idx % len(TAG_PALETTE)]
    short = tag.replace('_candidate','').replace('extragalactic_','ext_')

    # Nombre d'alertes par nuit
    n_alerts_per_night = df.groupby('night').size().sort_index()

    # Nombre d'objets nouveaux par nuit (première apparition)
    id_col = next((c for c in df.columns if 'diaObjectId' in c), None)
    if id_col:
        first_night = df.groupby(id_col)['night'].min()
        n_new_per_night = first_night.value_counts().sort_index()
    else:
        n_new_per_night = pd.Series(dtype=int)

    # ── Col 0 : alertes par nuit ───────────────────────────────────────────
    ax = axes[row_idx, 0]
    ax.bar(n_alerts_per_night.index, n_alerts_per_night.values,
           color=color, alpha=0.8, edgecolor='white', linewidth=0.3, width=0.8)
    ax.set_ylabel(f"{short}\nN alertes")
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)
    if row_idx == 0:
        ax.set_title('Alertes par nuit')

    # Annotation nuit max
    if not n_alerts_per_night.empty:
        peak_night = n_alerts_per_night.idxmax()
        peak_val   = n_alerts_per_night.max()
        ax.annotate(f"max : {peak_val}\n{peak_night.strftime('%Y-%m-%d')}",
                    xy=(peak_night, peak_val),
                    xytext=(8, 4), textcoords='offset points',
                    color=color,
                    arrowprops=dict(arrowstyle='->', color=color, lw=0.8))

    # ── Col 1 : objets nouveaux + courbe cumulative ────────────────────────
    ax  = axes[row_idx, 1]
    ax2 = ax.twinx()

    if not n_new_per_night.empty:
        ax.bar(n_new_per_night.index, n_new_per_night.values,
               color=color, alpha=0.5, edgecolor='white', linewidth=0.3, width=0.8,
               label='Nouveaux objets/nuit')
        cumul = n_new_per_night.sort_index().cumsum()
        ax2.plot(cumul.index, cumul.values, color='black', lw=1.5,
                 ls='-', label='Cumul objets')
        ax2.set_ylabel('N objets (cumulé)')
        ax2.tick_params(labelsize=8)

    ax.set_ylabel('Nouveaux objets/nuit')
    ax.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax.grid(True, axis='y', alpha=0.25, linewidth=0.5)
    if row_idx == 0:
        ax.set_title('Nouveaux objets / nuit + cumul')

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/nightcurve_per_tag.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/nightcurve_per_tag.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/nightcurve_per_tag.pdf / .png")

plt.show()

## 5. Décomposition par filtre nuit par nuit

In [ ]:
for tag in tags_ok:
    df = tag_data[tag].dropna(subset=['night'])
    if 'band' not in df.columns:
        continue

    short  = tag.replace('_candidate','').replace('extragalactic_','ext_')
    pivot  = df.groupby(['night','band']).size().unstack(fill_value=0)
    bands  = [b for b in ['u','g','r','i','z','y'] if b in pivot.columns]
    pivot  = pivot[bands]

    fig, (ax_stack, ax_heat) = plt.subplots(
        1, 2, figsize=(16, 4), layout='constrained')
    fig.suptitle(f"Distribution par filtre — {short}")

    # ── Barres empilées ────────────────────────────────────────────────────
    bottom = np.zeros(len(pivot))
    for band in bands:
        vals = pivot[band].values
        ax_stack.bar(pivot.index, vals, bottom=bottom,
                     color=FILTER_COLORS.get(band,'#888'),
                     label=f"${band}$", edgecolor='white', linewidth=0.2, width=0.8)
        bottom += vals
    ax_stack.set_ylabel('N alertes')
    ax_stack.xaxis.set_major_locator(mdates.AutoDateLocator())
    ax_stack.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.setp(ax_stack.xaxis.get_majorticklabels(), rotation=30, ha='right')
    ax_stack.legend(ncol=len(bands), loc='upper left')
    ax_stack.grid(True, axis='y', alpha=0.25, linewidth=0.5)
    ax_stack.set_title('Alertes empilées par filtre')

    # ── Heatmap nuit × filtre ──────────────────────────────────────────────
    # Normalisation par colonne (filtre) pour voir la répartition relative
    pivot_norm = pivot.div(pivot.max(axis=0).replace(0, 1), axis=1)
    im = ax_heat.imshow(pivot_norm.T.values, aspect='auto', cmap='YlOrRd',
                         origin='upper', interpolation='nearest')
    ax_heat.set_yticks(range(len(bands)))
    ax_heat.set_yticklabels([f"${b}$" for b in bands])

    # Ticks X : une sélection de dates
    n_dates = len(pivot)
    step    = max(1, n_dates // 10)
    tick_idx = range(0, n_dates, step)
    ax_heat.set_xticks(list(tick_idx))
    ax_heat.set_xticklabels(
        [pivot.index[i].strftime('%m-%d') for i in tick_idx],
        rotation=30, ha='right')
    fig.colorbar(im, ax=ax_heat, label='Activité relative (normalisée/filtre)')
    ax_heat.set_title('Heatmap nuit × filtre')

    if SAVE_FIGS:
        stem = f"{OUTPUT_DIR}/nightcurve_filters_{tag}"
        fig.savefig(f"{stem}.pdf", bbox_inches='tight', dpi=150)
        fig.savefig(f"{stem}.png", bbox_inches='tight', dpi=150)
        print(f"  ✓ {stem}.pdf / .png")

    plt.show()

## 6. Nuits les plus actives et gaps d'observation

In [ ]:
rows = []
for tag in tags_ok:
    df = tag_data[tag].dropna(subset=['night'])
    nightly = df.groupby('night').size().sort_index()
    if nightly.empty:
        continue

    dates_sorted = nightly.index.sort_values()
    gaps = (dates_sorted[1:] - dates_sorted[:-1]).days
    max_gap = int(gaps.max()) if len(gaps) > 0 else 0

    rows.append({
        'Tag'              : tag,
        'Première nuit'    : dates_sorted[0].strftime('%Y-%m-%d'),
        'Dernière nuit'    : dates_sorted[-1].strftime('%Y-%m-%d'),
        'N nuits actives'  : len(nightly),
        'Gap max (jours)'  : max_gap,
        'Nuit la + active' : nightly.idxmax().strftime('%Y-%m-%d'),
        'Max alertes/nuit' : int(nightly.max()),
        'Moy alertes/nuit' : round(nightly.mean(), 1),
    })

df_gaps = pd.DataFrame(rows)
html = df_gaps.to_html(index=False, escape=False, border=0, classes='fink-table')
style = """
<style>
  .fink-table { border-collapse: collapse; font-size: 12px; width: 100%; }
  .fink-table th { background: #f0f0f0; padding: 6px 10px;
                   border-bottom: 2px solid #ccc; text-align: left; }
  .fink-table td { padding: 4px 10px; border-bottom: 1px solid #eee; text-align: left; }
  .fink-table tr:hover td { background: #fafafa; }
</style>
"""
ipy_display(HTML(style + html))

## 7. Vue globale — tous tags sur le même axe temporel

In [ ]:
fig, ax = plt.subplots(figsize=(16, 5), layout='constrained')
fig.suptitle("Alertes par nuit — tous tags superposés")

for idx_t, tag in enumerate(tags_ok):
    df = tag_data[tag].dropna(subset=['night'])
    nightly = df.groupby('night').size().sort_index()
    if nightly.empty:
        continue
    color = TAG_PALETTE[idx_t % len(TAG_PALETTE)]
    short = tag.replace('_candidate','').replace('extragalactic_','ext_')
    ax.plot(nightly.index, nightly.values,
            color=color, lw=1.5, marker='o', ms=3, alpha=0.85, label=short)

ax.set_ylabel('N alertes par nuit')
ax.xaxis.set_major_locator(mdates.AutoDateLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
plt.setp(ax.xaxis.get_majorticklabels(), rotation=30, ha='right')
ax.legend(ncol=2, loc='upper left')
ax.grid(True, alpha=0.2, linewidth=0.5)

if SAVE_FIGS:
    fig.savefig(f"{OUTPUT_DIR}/nightcurve_all_tags.pdf", bbox_inches='tight', dpi=150)
    fig.savefig(f"{OUTPUT_DIR}/nightcurve_all_tags.png", bbox_inches='tight', dpi=150)
    print(f"✓ Sauvegardé : {OUTPUT_DIR}/nightcurve_all_tags.pdf / .png")

plt.show()